# Malaysian Car Plate YOLO Training

Train a one-class YOLO detector for `car plate` using the prepared dataset from this repository.

Before running: set Colab runtime to GPU.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure Paths

Upload `car_plate_yolo_colab.zip` to Google Drive, then update `ZIP_PATH` if needed.

In [ ]:
from pathlib import Path

ZIP_PATH = Path('/content/drive/MyDrive/capstone-alpr/car_plate_yolo_colab.zip')
DATASETS_DIR = Path('/content/datasets')
DATASET_ROOT = DATASETS_DIR / 'car_plate_yolo'
RUNS_DIR = Path('/content/drive/MyDrive/capstone-alpr/runs')

assert ZIP_PATH.exists(), f'Missing dataset zip: {ZIP_PATH}'
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import shutil

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
shutil.unpack_archive(str(ZIP_PATH), str(DATASETS_DIR))
file_count = sum(1 for path in DATASET_ROOT.rglob('*') if path.is_file())
print('files=', file_count)


In [ ]:
for split in ['train', 'val', 'test']:
    images = list((DATASET_ROOT / split / 'images').glob('*'))
    labels = list((DATASET_ROOT / split / 'labels').glob('*.txt'))
    print(split, 'images=', len(images), 'labels=', len(labels))
    assert images and labels
    assert len(images) == len(labels)

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
import ultralytics

print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
DATA_YAML = Path('/content/car_plate_data.yaml')
DATA_YAML.write_text(f'''path: {DATASET_ROOT}
train: train/images
val: val/images
test: test/images

nc: 1
names:
  0: car plate
''')
print(DATA_YAML.read_text())

## Train

This uses `yolo11n.pt` for the first prototype with 150 epochs, increased after the initial reported test performance of 92.8%.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=str(DATA_YAML),
    task='detect',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=20,
    seed=20260831,
    project=str(RUNS_DIR),
    name='yolov11n_car_plate_detection',
    exist_ok=True,
)

In [ ]:
best_weights = RUNS_DIR / 'yolov11n_car_plate_detection' / 'weights' / 'best.pt'
assert best_weights.exists(), best_weights
print(best_weights)

In [ ]:
trained = YOLO(str(best_weights))
val_metrics = trained.val(data=str(DATA_YAML), split='val', imgsz=640, project=str(RUNS_DIR), name='val_metrics', exist_ok=True)
test_metrics = trained.val(data=str(DATA_YAML), split='test', imgsz=640, project=str(RUNS_DIR), name='test_metrics', exist_ok=True)
print('Validation metrics:', val_metrics.results_dict)
print('Test metrics:', test_metrics.results_dict)

In [ ]:
export_path = trained.export(format='onnx', imgsz=640)
print(export_path)

In [ ]:
from google.colab import files

files.download(str(best_weights))